In [1]:
import os
import pandas as pd
import torch
from ARIAdataset import buildARIA
from torch.utils.data import random_split
#from AiArtBench import buildArtBenchDataset as buildArtBench

# Helper notebook to generate CSVs of desired dataset.
# simple helper functions to generate CSVs of desired folders for model inference.

root = '/path/goes/here/' # change to what you need.

original_csvs = ['DALL-E', 'Midjourney', 'StarryAI', 'DreamStudio']
   

def create_ARIA_CSV_test(seed=42): 
    aria = buildARIA()
    # /home/.../ARIA_dataset/generator/method/category/img.jpg

    trainSize = int(len(aria) * 0.8)
    testSize = len(aria) - trainSize

    _, testAria = random_split(
        aria,
        [trainSize, testSize],
        generator=torch.manual_seed(seed)
    )

    rows = []
    count = 0

    for global_idx in testAria.indices:
        running = 0

        for ds in aria.datasets:
            if global_idx < running + len(ds):
                local_idx = global_idx - running

                img_path = ds.image_paths[local_idx]
                label = ds.label

                parts = img_path.replace("\\", "/").split("/")

                generator = "real"
                method = "real"
                category = "real"

                if "ARIA_dataset" in parts:
                    idx = parts.index("ARIA_dataset")
                    remaining = parts[idx + 1:]

                    # REAL structure: Real/<category>/img
                    if remaining[0].lower() == "real":
                        generator = "real"
                        method = "real"
                        category = remaining[1] if len(remaining) > 1 else "unknown"

                    # AI structure: gen/method/category/img
                    elif len(remaining) >= 4:
                        generator = remaining[0]
                        method = remaining[1]
                        category = remaining[2]

                rows.append({
                    "image": img_path,
                    "generator": generator,
                    "method": method,
                    "category": category,
                    "class": label,
                    "split": "test"
                })

                count += 1
                if count % 250 == 0:
                    print(f"{count} rows added")

                break

            running += len(ds)

    return pd.DataFrame(rows)


def create_CSV_from_CSV(csv):
    df = pd.read_csv(f"{root}{csv}.csv")

    # filter categories
    filtered_df = df[(df["Category"] == "art") | (df["Category"] == "pixiv")].copy()

    # build lookup: stem (no extension) -> full path
    image_lookup = {}

    valid_exts = (".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tiff")

    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if fname.lower().endswith(valid_exts):
                stem = os.path.splitext(fname)[0]  # remove extension
                image_lookup[stem] = os.path.join(dirpath, fname)

    # map image_name (no extension) -> full path
    filtered_df["image"] = filtered_df["Image_Name"].map(image_lookup)

    # drop missing matches
    filtered_df = filtered_df.dropna(subset=["image"])

    # build final required structure
    filtered_df = pd.DataFrame({
        "image": filtered_df["image"],
        "class": 1,
        "split": "test"
    })

    return filtered_df

ModuleNotFoundError: No module named 'ARIAdataset'

In [3]:
import pandas as pd

def create_artbench_CSV():
    root_dir = '/home/stu14/s16/acm7552/Capstone/data/Real_AI_SD_LD_Dataset/test'
    rows = []

    for folder in os.listdir(root_dir):
        folder_path = os.path.join(root_dir, folder)

        if not os.path.isdir(folder_path):
            continue

        # determine class
        is_ai = folder.startswith("AI")
        cls = 1 if is_ai else 0

        # determine method
        if is_ai:
            if "_SD_" in folder:
                method = "SD"
            elif "_LD_" in folder:
                method = "LD"
            else:
                raise ValueError(f"Unknown method in folder: {folder}")
        else:
            method = "real"

        # iterate images
        for file in os.listdir(folder_path):
            if file.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".webp")):
                img_path = os.path.join(folder_path, file)

                rows.append({
                    "image": img_path,
                    "class": cls,
                    "method": method,
                    "split": "test"
                })

    df = pd.DataFrame(rows)
    
    return df

In [ ]:
# create_CSV_from_CSV('DALL-E')

In [ ]:
# df = pd.read_csv(f'{root}{original_csvs[0]}.csv')
# df

In [ ]:
#filtered_df = df[(df['Category'] == 'art') | (df['Category'] == 'pixiv')]
# filtered_df

In [ ]:
#aria_df = create_ARIA_CSV_test(42)
#aria_df['category'] = aria_df['category'].replace('art_5400', 'art')
#aria_df['category'] = aria_df['category'].replace('pixiv_3k', 'pixiv')
#aria_df

In [ ]:
# reals = aria_df[aria_df['class']==0]
# reals

In [ ]:
#aria_df.to_csv('ariaTest.csv')

In [4]:
artBench = create_artbench_CSV()
artBench

,image,class,method,split
0,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,0,real,test
1,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,0,real,test
2,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,0,real,test
3,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,0,real,test
4,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,0,real,test
...,...,...,...,...
29995,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,1,SD,test
29996,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,1,SD,test
29997,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,1,SD,test
29998,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,1,SD,test


In [6]:
# test set is 30,000 images. if i need to make it smaller for inference, do this

def sample_per_chunk(df, chunk_size=1000, sample_size=200, seed=42):
    sampled = []

    for i in range(0, len(df), chunk_size):
        chunk = df.iloc[i:i+chunk_size]
        sampled.append(chunk.sample(n=min(sample_size, len(chunk)), random_state=seed))

    return pd.concat(sampled).reset_index(drop=True)

In [7]:
sampled_artbench = sample_per_chunk(artBench)
sampled_artbench

,image,class,method,split
0,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,0,real,test
1,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,0,real,test
2,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,0,real,test
3,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,0,real,test
4,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,0,real,test
...,...,...,...,...
5995,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,1,SD,test
5996,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,1,SD,test
5997,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,1,SD,test
5998,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,1,SD,test


In [8]:
sampled_artbench.to_csv('artbenchTest.csv')